In [1]:
%pip install semchunk

Note: you may need to restart the kernel to use updated packages.


In [2]:
# Download and clean arXiv sources, keeping the papers in memory.
from pathlib import Path
import sys

import requests

PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / "paper_clustering").is_dir():
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from paper_clustering.extract.extract_text import get_paper
from scripts.label_sentences import ParagraphSample, normalize_arxiv_id

# Populate this list, or define arxiv_ids in an earlier cell.
arxiv_ids = globals().get("arxiv_ids", ["1808.01137", "1301.1873"])
INCLUDE_PROOFS = False
if not arxiv_ids:
    raise ValueError("Populate arxiv_ids with the papers to inspect.")
arxiv_ids = list(dict.fromkeys(normalize_arxiv_id(value) for value in arxiv_ids))

papers = {}
with requests.Session() as session:
    for arxiv_id in arxiv_ids:
        print(f"Downloading {arxiv_id}...")
        papers[arxiv_id] = get_paper(session, arxiv_id, include_proofs=INCLUDE_PROOFS)
print(f"Loaded {len(papers)} papers.")

Loaded 2 papers.


In [3]:
# Semantically chunk cleaned sections and score the top five chunks per paper.
import semchunk
import torch

from paper_clustering.technique_classifier import SciBERTClassifier

CLASSIFIER_WEIGHTS = PROJECT_ROOT / SciBERTClassifier.DEFAULT_WEIGHTS_PATH
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
POOLING = 'max'  # Default truncation; use 'mean' or 'max' for long passages.
classifier = SciBERTClassifier(weights_path=CLASSIFIER_WEIGHTS).to(DEVICE)
classifier.eval()

# Reserve room for SciBERT's special tokens within its 512-token input.
CHUNK_SIZE = 768 - classifier.tokenizer.num_special_tokens_to_add(pair=False)
chunker = semchunk.chunkerify(
    classifier.tokenizer, chunk_size=CHUNK_SIZE,
    tokenizer_kwargs={"add_special_tokens": False},
)

# Scores are expected ordinal labels from 0 to 4, not probabilities.
top_passages = {}
for arxiv_id, paper in papers.items():
    print(f"\n{arxiv_id}: {paper.metadata.title}")
    passages = [
        ParagraphSample(
            arxiv_id=arxiv_id,
            section_index=section.section_index,
            paragraph_index=chunk_index,  # Index of the chunk within this section.
            section_header=section.section_header or "(Untitled section)",
            paragraph=chunk,
        )
        for section in paper.sections
        for chunk_index, chunk in enumerate(chunker("\n".join(section.text)))
        if chunk.strip()
    ]
    if not passages:
        top_passages[arxiv_id] = []
        print("No passages could be extracted from this paper's source.")
        continue
    scores = classifier.predict([p.paragraph for p in passages], pooling=POOLING)
    ranked = sorted(zip(passages, scores.tolist()), key=lambda item: item[1], reverse=True)[:5]
    top_passages[arxiv_id] = ranked
    for rank, (passage, score) in enumerate(ranked, start=1):
        print(f"\n{rank}. Score: {score:.4f} / 4 | {passage.section_header}")
        print(passage.paragraph)


1808.01137: When Does Hillclimbing Fail on Monotone Functions: An entropy compression argument


Evaluating: 100%|██████████| 1/1 [00:00<00:00,  1.26it/s]



1. Score: 3.5684 / 4 | Introduction
We first give some intuition on the behaviour of the algorithm and on our proof. For a general monotone function f, it is natural to measure the progress of the (1+1) -EA is terms of number of one-bits in the current search point. In order to make an update, it is necessary to flip at least one zero-bit into a one-bit, since otherwise the offspring would be rejected due to monotonicity (unless it is identical to the parent, in which case there is no update either). Thus, if the average update does not flip too many ones to zeros, the number of ones in the current search point will tend to n efficiently. For a small mutation parameter (specifically for c<1 ), this is indeed true as the average number of ones flipped to zeros is at most c, and this remains true for update steps. For larger c, one might expect this to still hold as, intuitively, any offspring with more ones flipped to zeros than zeros flipped to ones should be unlikely to be fitter tha

Evaluating: 100%|██████████| 1/1 [00:00<00:00,  1.17it/s]


1. Score: 2.7373 / 4 | Bound on  g_4(ell)  for  24le ellle 99 
The second method to bound the size of g_4(ell) is based on ordinary generating functions (OGF). Here, k=4, so let A_1, A_2, A_3, A_4 be the four variables of p and let a_i be the number of apparitions of A_i in p. Therefore, a_1+a_2+a_3+a_4=|p|. Recall that each variable appears at least twice in p since p is doubled, so a_ige 2. Moreover, a factor of length ell, with 24leellle 99, is necessarily an occurrence of a pattern of length between 24 and 99. So we just have to consider patterns p with 24le|p|le 99
Given L'= L_1, L_2, L_3 an element of L that corresponds to some factor f_i, we can compute the lengths ell_i of each variable A_i in f_i ( ell_1 = L_1, ell_i = L_i-L_(i-1) for iin 2, 3 and ell_4 = |f_i|-(a_1ell_1+a_2ell_2+a_3ell_3) (a_4) ). Recall that ell_ige 1 since each variable of p is a non-empty word. Let A _(|p|)=sum_(ige |p|)b_i x^i be the OGF of such sets L', i. e. b_i is the number of 3 -sets L_1, L_2, L_3 t

In [5]:
# Load Nomic once and compare two passages using normalized embeddings.
from sentence_transformers import SentenceTransformer

from paper_clustering.embedding import embed

EMBEDDING_MODEL = "nomic-ai/nomic-embed-text-v2-moe"
embedding_model = SentenceTransformer(EMBEDDING_MODEL, trust_remote_code=True, device=DEVICE)


def passage_inner_product(first: str, second: str) -> float:
    """Return the inner product of unit embeddings (cosine similarity)."""
    vectors = embed([first, second], embedding_model, device=DEVICE)
    return float(vectors[0] @ vectors[1])


# Example: passage_inner_product("First passage", "Second passage")
s1 = """We first give some intuition on the behaviour of the algorithm and on our proof. For a general monotone function f, it is natural to measure the progress of the (1+1) -EA is terms of number of one-bits in the current search point. In order to make an update, it is necessary to flip at least one zero-bit into a one-bit, since otherwise the offspring would be rejected due to monotonicity (unless it is identical to the parent, in which case there is no update either). Thus, if the average update does not flip too many ones to zeros, the number of ones in the current search point will tend to n efficiently. For a small mutation parameter (specifically for c<1 ), this is indeed true as the average number of ones flipped to zeros is at most c, and this remains true for update steps. For larger c, one might expect this to still hold as, intuitively, any offspring with more ones flipped to zeros than zeros flipped to ones should be unlikely to be fitter than its parent. However, the reason this intuition fails for sufficiently large c is that one can "trick" the algorithm by weighing a fraction of the remaining zeros much higher than most ones already in the search point. Then, whenever one of these zeroes is flipped, the algorithm will happily keep the offspring, regardless of how many ones are flipped to zeros in the process
Based on this intuition, we define good and bad updates. The bad updates capture cases in which a zero-bit with a disproportionally high weight is flipped. Then we study the entropy of the update steps of the algorithms. On the one hand, we will analyze the algorithm in a forward manner to give a lower bound on the entropy of each update. On the other hand, we give a backwards encoding (from last step to first) of the updates steps, and this encoding saves some bits in bad update steps. Since the expected number of bits needed for the encoding is lower bounded by the entropy, we get an upper bound for the expected number of bad update steps. This, in turn, gives us a linear upper bound for the expected number of update steps. Finally, the runtime bound follows by a slight refinement of the calculation, in which we compute how many steps we need to decrease the number of zero-bits from 2^k to 2^(k-1), for k=log n, dots, 1
"""
s2 = """An important part of the algorithm is to keep the record R of each step of the algorithm. Let R_i be the record after i steps of the algorithm. On one hand, given V as input of the algorithm, this produces a pair (R_t, w_t). On the other hand, given a pair (R_t, w_t), we will show in Lemma that we can recover the entire input vector V. So, each input vector V produces a distinct pair (R_t, w_t). Let V be the set of input vectors V of size t, let R be the set of records R produced by the algorithm and let O be the set of different outputs (R_t, w_t). After the execution of the algorithm ( t steps), w_t avoids p by definition and therefore |w_t|<n by contradiction hypothesis. Hence, the number of possible final words w_t is independent from t (it is at most 2^n ). We then clearly have | O | le 2^ntimes | R |. We will prove that | V |le | O | and that | R |=o(2^t) to obtain the contradiction 2^t=| V |le | O | le 2^ntimes| R |=o(2^t)
The record R is a triplet R=(D, L, X) where D is a binary word (each element is 0 or 1 ), L is a vector of (k-1) -sets of non-zero integers and X is a vector of binary words. At the beginning, D, L and X are empty. At step i of the algorithm, we append V[i] to w_(i-1) to get w'_i
If w'_i contains no occurrence of p, then we append 0 to D to get R_i and we set w_i=w'_i. Otherwise, suppose that w'_i contains an occurrence y of p that forms a factor f of length ell ( f is the ell last letters of w'_i ). Recall that A_j is the j -th variable of p. Let ell_j=|A_j| in the factor f, let L_1 = ell_1, L_j=L_(j-1)+ell_j for jge 2. Let L'=set(L_1, L_2, ldots, L_(k-1)) be a (k-1) -set of non-zero integers. Let X' be the binary word obtained from A_1cdot A_2cdotldotscdot A_k (where " cdot " is the concatenation operator) followed by as many 0 's as necessary to get length 2. Note that we necessarily have |A_1cdot A_2cdotldotscdot A_k|le 2 since the pattern is doubled. Eventually, to get R_i, we append the factor 01^ell to D; we add L' as the last element of L; finally we add X' as the last element of X
"""
passage_inner_product(s1, s2)

/home/alfred/.cache/huggingface/modules/transformers_modules/nomic_hyphen_ai/nomic_hyphen_bert_hyphen_2048/7710840340a098cfb869c4f65e87cf2b1b70caca/modeling_hf_nomic_bert.py:1634: UserWarning: Install Nomic's megablocks fork for better speed: `pip install git+https://github.com/nomic-ai/megablocks.git`
  warnings.warn("Install Nomic's megablocks fork for better speed: " +


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

0.6474536061286926